In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import argparse

In [ ]:
version = 8

n = 100
# subset event level
# fp = f"./365day_future_prediction_outputs_50_subset_{n}_stage_filter_v{version}"
# full event level
# fp = f"./365day_future_prediction_outputs_50_full_stage_filter_v{version}"

# subset patient level
# fp = "./365day_future_prediction_outputs_50_subset_1000_stage_filter_patient_level_v2"
# full patient level 
fp = "./365day_future_prediction_outputs_50_full_stage_filter_patient_level_v2"
print(fp)

In [ ]:
dirs_1 = [
    "/LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/MLP_365DayFutureTarget_detailed_outputs.csv",
    "/RNN_365DayFutureTarget_detailed_outputs.csv",
    "/TCN_365DayFutureTarget_detailed_outputs.csv",
    "/Transformer_365DayFutureTarget_detailed_outputs.csv",
]

dirs_2 = [
    "/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_RNN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_TCN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_Transformer_365DayFutureTarget_detailed_outputs.csv",
]

dirs = dirs_1 + dirs_2

filepaths = [fp + i for i in dirs]
filepaths

In [ ]:
# A global dictionary to keep IDs consistent across multiple function calls
patient_registry = {}

def mask_to_sequence(df, column="PatientID"):
    """
    Maps unique Patient IDs to a sequence from 1 to N.
    Maintains consistency across different DataFrames.
    """
    global patient_registry
    
    # 1. Identify IDs in this table that we haven't seen before
    new_ids = [pid for pid in df[column].unique() if pid not in patient_registry]
    
    # 2. Assign the next available numbers to these new IDs
    current_max = len(patient_registry)
    for i, pid in enumerate(new_ids):
        patient_registry[pid] = current_max + i + 1
        
    # 3. Create the masked copy
    df_masked = df.copy()
    df_masked[column] = df_masked[column].map(patient_registry)
    
    return df_masked

In [ ]:
new_fp = fp + "_masked"
print(new_fp)
os.makedirs(new_fp, exist_ok=True) 

for filepath in filepaths:
    print(filepath)
    fn = os.path.basename(filepath)

    split_fn = fn.split(".")
    new_fn = split_fn[0] + "_masked." + split_fn[1]
    # new_fn = "masked_" + fn
    print(new_fn)

    df = pd.read_csv(filepath)
    df_masked = mask_to_sequence(df)

    new_out = os.path.join(new_fp, new_fn)
    print(new_out)    
    
    df_masked.to_csv(new_out, index=False)
    # break


In [ ]:
print(new_fp)

In [ ]:
os.listdir(new_fp)

In [ ]:
test_file = os.listdir(new_fp)[-1]
print(test_file)
check = os.path.join(new_fp, test_file)
pd.read_csv(check)

In [ ]:
# sanity check
for i in os.listdir(new_fp):
    print(pd.read_csv(os.path.join(new_fp, i)).head(2)['PatientID'])
    print(pd.read_csv(os.path.join(new_fp, i)).tail(2)['PatientID'])